# Week 6 — greedy reversals and approximation ratios

Open in VS Code. **Select Kernel** (Python 3). **Run All** from the top. Needs `matplotlib`.

Same examples as the lecture: example 1 $\pi = [6,1,2,3,4,5]$, example 2 $\pi = [4,5,6,1,2,3]$.

**What to do**

1. Place-one-block rule on example 1 → 5 reversals.
2. Strips algorithm on example 1 (2 reversals) and example 2 (3 reversals).
3. Example 2: try all 15 reversals; none lowers $b$.
4. Sample random permutations; compare $A/\mathrm{OPT}$ with $A/(b/2)$.
5. Write Q1–Q3 below, without notes.

In [ ]:
from collections import deque
import random

import matplotlib.pyplot as plt


def rev(pi, i, j):
    """Reverse positions i..j of pi (1-based, inclusive)."""
    pi = list(pi)
    pi[i - 1:j] = pi[i - 1:j][::-1]
    return pi


def with_ends(pi):
    return [0] + list(pi) + [len(pi) + 1]


def breakpoints(pi):
    q = with_ends(pi)
    return sum(abs(q[k] - q[k + 1]) != 1 for k in range(len(q) - 1))


def show(pi):
    """Print pi with end markers and a bar at every breakpoint."""
    q = with_ends(pi)
    out = str(q[0])
    for a, b in zip(q, q[1:]):
        out += (" " if abs(a - b) == 1 else " | ") + str(b)
    return out


toy1 = [6, 1, 2, 3, 4, 5]
toy2 = [4, 5, 6, 1, 2, 3]
print(show(toy1), "   b =", breakpoints(toy1))
print(show(toy2), "   b =", breakpoints(toy2))

## Put the next block in place

Find the first position $i$ that does not hold block $i$. Reverse from $i$ to wherever block $i$ sits.

In [ ]:
def place_blocks(pi, verbose=True):
    pi = list(pi)
    flips = 0
    for i in range(1, len(pi) + 1):
        j = pi.index(i) + 1
        if j != i:
            pi = rev(pi, i, j)
            flips += 1
            if verbose:
                print(f"step {flips}:", show(pi), "   b =", breakpoints(pi))
    return flips


A_home = place_blocks(toy1)
print("place-one-block A =", A_home)

## Strips and the $k-1$ rule

A strip is a run between bars. A lone block counts as decreasing; strips holding the end markers count as increasing.

If a decreasing strip exists: take the smallest block $k$ in any decreasing strip and reverse so that it lands next to $k-1$. Otherwise reverse an increasing strip.

In [ ]:
def strips(pi):
    """List of (strip, is_decreasing), end markers included."""
    q = with_ends(pi)
    runs, cur = [], [q[0]]
    for a, b in zip(q, q[1:]):
        if abs(a - b) == 1:
            cur.append(b)
        else:
            runs.append(cur)
            cur = [b]
    runs.append(cur)
    end = len(pi) + 1
    out = []
    for s in runs:
        if 0 in s or end in s:
            dec = False
        elif len(s) == 1:
            dec = True
        else:
            dec = s[1] < s[0]
        out.append((s, dec))
    return out


def strips_step(pi):
    dec = [s for s, d in strips(pi) if d]
    if not dec:
        # step 4: reverse the first increasing strip that is not an end marker
        end = len(pi) + 1
        s = next(s for s, d in strips(pi) if 0 not in s and end not in s)
        i = pi.index(s[0]) + 1
        return rev(pi, i, i + len(s) - 1)
    # step 3: put k next to k-1
    k = min(min(s) for s in dec)
    pk = pi.index(k) + 1
    pm = pi.index(k - 1) + 1 if k > 1 else 0
    return rev(pi, pm + 1, pk) if pm < pk else rev(pi, pk + 1, pm)


def strips_sort(pi, verbose=True):
    pi = list(pi)
    flips = 0
    if verbose:
        print("start: ", show(pi), "   b =", breakpoints(pi))
    while breakpoints(pi) > 0:
        pi = strips_step(pi)
        flips += 1
        if verbose:
            print(f"step {flips}:", show(pi), "   b =", breakpoints(pi))
    return flips


print("example 1, A =", strips_sort(toy1))
print()
print("example 2, A =", strips_sort(toy2))

## Example 2 is stuck

Try every reversal of example 2. Does any lower $b$ below 3?

In [ ]:
n = len(toy2)
for i in range(1, n + 1):
    for j in range(i + 1, n + 1):
        q = rev(toy2, i, j)
        print(f"rev[{i}..{j}]", show(q), "   b =", breakpoints(q))

## Exact OPT on small permutations

Breadth-first search over all reversals gives the true fewest flips. Fine for $n \le 7$.

In [ ]:
def opt(pi):
    start = tuple(pi)
    target = tuple(sorted(pi))
    dist = {start: 0}
    queue = deque([start])
    while queue:
        cur = queue.popleft()
        if cur == target:
            return dist[cur]
        n = len(cur)
        for i in range(1, n + 1):
            for j in range(i + 1, n + 1):
                nxt = tuple(rev(cur, i, j))
                if nxt not in dist:
                    dist[nxt] = dist[cur] + 1
                    queue.append(nxt)


for name, pi in [("example 1", toy1), ("example 2", toy2)]:
    print(name, "  b/2 =", breakpoints(pi) / 2, "  OPT =", opt(pi))
print("place-one-block r on example 1 =", A_home / opt(toy1))

## Instance ratios on random permutations

For each random permutation: $A/\mathrm{OPT}$ (exact) and $A/(b/2)$ (needs no OPT). Since $\mathrm{OPT} \ge b/2$, the second is never smaller than the first.

In [ ]:
random.seed(6)


def sample(n, N):
    rows = []
    for _ in range(N):
        pi = list(range(1, n + 1))
        random.shuffle(pi)
        b = breakpoints(pi)
        if b == 0:
            continue
        o = opt(pi)
        for rule, f in [("place-one-block", place_blocks), ("strips", strips_sort)]:
            A = f(pi, verbose=False)
            rows.append({"rule": rule, "r": A / o, "r_bound": A / (b / 2)})
    return rows


rows = sample(6, 300)
for rule in ["place-one-block", "strips"]:
    r = [x["r"] for x in rows if x["rule"] == rule]
    rb = [x["r_bound"] for x in rows if x["rule"] == rule]
    print(f"{rule:9s}  A/OPT: mean={sum(r)/len(r):.2f} max={max(r):.2f}"
          f"   A/(b/2): mean={sum(rb)/len(rb):.2f} max={max(rb):.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), sharey=True)
for ax, key, label in [(axes[0], "r", "$A/\\mathrm{OPT}$"), (axes[1], "r_bound", "$A/(b/2)$")]:
    for rule, col in [("place-one-block", "#2a5a8c"), ("strips", "#b56a3a")]:
        vals = [x[key] for x in rows if x["rule"] == rule]
        ax.hist(vals, bins=12, alpha=0.55, color=col, label=rule)
    ax.axvline(4.0, color="#1c2430", linestyle="--", linewidth=1.2, label="strips guarantee 4")
    ax.set_xlabel(label)
axes[0].set_ylabel("count")
axes[0].legend()
fig.suptitle("Random permutations, $n=6$: instance ratios")
plt.tight_layout()
plt.show()

## Questions

Without looking at notes. **Q1–Q2:** today’s idea. **Q3:** what the histogram says.

**Q1.** The place-one-block rule uses 5 reversals on example 1, while the optimum is 2. Describe the rule, and give $r$ for this input.

A1.

**Q2.** Why can a single reversal remove at most two breakpoints? How does this, together with the strips algorithm, give an approximation ratio of at most 4?

A2.

**Q3.** The maximum of $A/(b/2)$ is larger than its mean, and larger than the maximum of $A/\mathrm{OPT}$. Why? What can such a histogram tell us about the approximation ratio, and what can it not?

A3.